# Tarapath Position Solver Validation

This notebook validates the core celestial navigation math used by Tarapath.
It uses a few bright stars with known right ascension (RA) and declination (Dec),
a known observer location and timestamp, and checks that the grid-search
location solver can recover the original location from synthetic observations.

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u


@dataclass
class StarObservation:
    name: str
    ra_deg: float
    dec_deg: float
    observed_alt_deg: float
    observed_az_deg: float


def compute_alt_az(ra_deg: float, dec_deg: float, lat_deg: float, lon_deg: float, timestamp: Time) -> Tuple[float, float]:
    coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    location = EarthLocation(lat=lat_deg * u.deg, lon=lon_deg * u.deg)
    altaz_frame = AltAz(obstime=timestamp, location=location)
    altaz = coord.transform_to(altaz_frame)
    return float(altaz.alt.deg), float(altaz.az.deg)


def estimate_location(
    observed_stars: List[StarObservation],
    timestamp: Time,
    lat_bounds: Tuple[float, float],
    lon_bounds: Tuple[float, float],
    step_deg: float,
) -> Tuple[float, float, float]:
    """Brute-force grid search over lat/lon to find best match.

    Returns (best_lat, best_lon, best_error).
    """
    best_error = float("inf")
    best_lat, best_lon = None, None

    lat_vals = np.arange(lat_bounds[0], lat_bounds[1] + step_deg, step_deg)
    lon_vals = np.arange(lon_bounds[0], lon_bounds[1] + step_deg, step_deg)

    for lat in lat_vals:
        for lon in lon_vals:
            err2_sum = 0.0
            for star in observed_stars:
                pred_alt, pred_az = compute_alt_az(star.ra_deg, star.dec_deg, lat, lon, timestamp)
                d_alt = pred_alt - star.observed_alt_deg
                d_az = pred_az - star.observed_az_deg
                err2_sum += d_alt ** 2 + d_az ** 2
            if err2_sum < best_error:
                best_error = err2_sum
                best_lat, best_lon = lat, lon

    return best_lat, best_lon, best_error


In [ ]:
# Define a known location and time
true_lat, true_lon = 37.7749, -122.4194  # San Francisco approx
timestamp = Time("2024-03-15T10:00:00", scale="utc")

# A few bright stars (approximate RA/Dec in degrees)
catalog = [
    ("Sirius", 101.287, -16.716),
    ("Canopus", 95.987, -52.696),
    ("Arcturus", 213.915, 19.182),
]

# Generate synthetic observations at the true location
observed: List[StarObservation] = []
for name, ra, dec in catalog:
    alt, az = compute_alt_az(ra, dec, true_lat, true_lon, timestamp)
    # Add small measurement noise
    alt_obs = alt + np.random.normal(0, 0.05)
    az_obs = az + np.random.normal(0, 0.05)
    observed.append(StarObservation(name=name, ra_deg=ra, dec_deg=dec, observed_alt_deg=alt_obs, observed_az_deg=az_obs))

# Run the grid search around the true location
lat_bounds = (true_lat - 5, true_lat + 5)
lon_bounds = (true_lon - 5, true_lon + 5)

best_lat, best_lon, best_error = estimate_location(observed, timestamp, lat_bounds, lon_bounds, step_deg=0.5)
best_lat, best_lon, best_error